In [1]:
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env.

import os, json, datetime
from crossref_provider import search_crossref
from enhanced_query_script import to_jsonl, to_csv  # your existing utils

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
outdir = os.path.join("../outputs", f"crossref_run_{timestamp}")
os.makedirs(outdir, exist_ok=True)

with open("../queries/queries_crossref.json", "r", encoding="utf-8") as f:
    queries = json.load(f)

for idx, q in enumerate(queries, start=1):
    print(f"\n=== Crossref Query {idx}/{len(queries)} ===")
    print(q)
    results = search_crossref(
        q,
        year_min=2018,
        mailto=os.getenv("CROSSREF_EMAIL"),
        types=["journal-article", "proceedings-article", "posted-content"],
        has_abstract=None,              # set True to require abstracts
        rows=200,
        sort="is-referenced-by-count",  # or 'published' or 'score'
        order="desc",
        polite_delay=1.0
    )
    for r in results:
        r["query"] = q
        r["query_id"] = f"CR{idx:02d}"
    to_jsonl(os.path.join(outdir, f"CR{idx:02d}_results.jsonl"), results)
    to_csv  (os.path.join(outdir, f"CR{idx:02d}_results.csv"),   results)
    print(f"Found {len(results)} results")




=== Crossref Query 1/16 ===
"field conditions" plant disease pest image imaging vision camera rgb hyperspectral uav drone adaptation domain shift generalization robustness
Crossref request: https://api.crossref.org/works?query=%22field+conditions%22+plant+disease+pest+image+imaging+vision+camera+rgb+hyperspectral+uav+drone+adaptation+domain+shift+generalization+robustness&rows=200&cursor=%2A&filter=from-pub-date%3A2018-01-01%2Ctype%3Ajournal-article%2Ctype%3Aproceedings-article%2Ctype%3Aposted-content&mailto=bekhouche.mouadh%40univ-oeb.dz&select=DOI%2Ctitle%2Cauthor%2Ccontainer-title%2Cissued%2Ctype%2CURL%2Cis-referenced-by-count%2Clink%2Cabstract%2Csubject%2Cpublisher%2Clicense
Crossref request: https://api.crossref.org/works?query=%22field+conditions%22+plant+disease+pest+image+imaging+vision+camera+rgb+hyperspectral+uav+drone+adaptation+domain+shift+generalization+robustness&rows=200&cursor=DnF1ZXJ5VGhlbkZldGNoJAAAAAAnq0UrFnV2UkozOWtFU2oyNm5wNjdLSVRnWWcAAAAAJ8-8hRZORTZQQUE5MVNyMnlT

In [3]:
# aggregate all results
import glob
all_results = []
for fn in glob.glob(os.path.join(outdir, "CR??_results.jsonl")):
    with open(fn, "r", encoding="utf-8") as f:
        for line in f:
            all_results.append(json.loads(line))
print(f"Total aggregated results: {len(all_results)}")
to_jsonl(os.path.join(outdir, f"all_results.jsonl"), all_results)
to_csv  (os.path.join(outdir, f"all_results.csv"),   all_results)

Total aggregated results: 6400
